In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("..\data\processed\cases.csv")



In [10]:
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   case_id          40 non-null     int64  
 1   nomor_perkara    40 non-null     object 
 2   pengadilan       40 non-null     object 
 3   penggugat        40 non-null     object 
 4   tergugat         40 non-null     object 
 5   objek_sengketa   40 non-null     object 
 6   luas_tanah       40 non-null     float64
 7   ringkasan_fakta  35 non-null     object 
 8   amar_putusan     35 non-null     object 
 9   label            40 non-null     object 
 10  text_full        40 non-null     object 
dtypes: float64(1), int64(1), object(9)
memory usage: 3.6+ KB
None
   case_id      nomor_perkara  \
0        1  101/Pdt.G/2020/PN   
1        2  109/Pdt.G/2020/PN   
2        3  115/Pdt.G/2020/PN   
3        4  143/Pdt.G/2020/PN   
4        5  148/Pdt.G/2020/PN   

                                 

In [11]:
print(df.isnull().sum())

case_id            0
nomor_perkara      0
pengadilan         0
penggugat          0
tergugat           0
objek_sengketa     0
luas_tanah         0
ringkasan_fakta    5
amar_putusan       5
label              0
text_full          0
dtype: int64


In [12]:
print(df["label"].value_counts())

label
Dikabulkan Sebagian    33
Lainnya                 5
Dikabulkan              2
Name: count, dtype: int64


In [13]:
df["ringkasan_fakta"]

0      Menimbang, bahwa Penggugat dengan surat gugat...
1      Menimbang, bahwa Penggugat dengan surat gugat...
2      Menimbang, bahwa Penggugat dengan surat gugat...
3      n A Menimbang, bahwa Penggugat dengan surat g...
4                                                   NaN
5      n AMenimbang, bahwa Penggugat dengan surat gu...
6      d g Menimbang, bahwa Penggugat dengan surat g...
7      d g Menimbang, bahwa Penggugat dengan surat g...
8      d g Menimbang, bahwa Penggugat dengan surat g...
9      d g Menimbang, bahwa Penggugat dengan surat g...
10     d g Menimbang, bahwa Penggugat dengan surat g...
11     Menimbang, bahwa Penggugat dengan surat gugat...
12     d g Menimbang, bahwa Penggugat dengan surat g...
13     o u Menimbang, bahwa Penggugat dengan surat g...
14     d g Menimbang, bahwa Penggugat dengan surat g...
15     o u Menimbang, bahwa Penggugat dengan surat g...
16                                                  NaN
17     n n Menimbang, bahwa Penggugat dengan sur

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words=None
)

docs = df["ringkasan_fakta"].fillna("").astype(str)
X = vectorizer.fit_transform(docs)

print(X.shape)
print("Nulls in ringkasan_fakta:", df["ringkasan_fakta"].isna().sum())


(40, 447)
Nulls in ringkasan_fakta: 5


In [15]:
print(
    vectorizer.get_feature_names_out()[:50]
)

['000' '00438' '00444' '00454' '00470' '00491' '00496' '00527' '00632'
 '00651' '00654' '00657' '00680' '00686' '00696' '00712' '00724' '00733'
 '00738' '00768' '00769' '00782' '00874' '00893' '00899' '00931' '00941'
 '00964' '00966' '00989' '00991' '01007' '01010' '01016' '01024' '01026'
 '01041' '01057' '01071' '01150' '01173' '01176' '01183' '01185' '01197'
 '01206' '01223' '01231' '01249' '01258']


In [16]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, k=5):

    query_vector = vectorizer.transform([query])

    similarity = cosine_similarity(
        query_vector,
        X
    )

    top_idx = np.argsort(
        similarity[0]
    )[-k:][::-1]

    return df.iloc[top_idx][[
        "case_id",
        "nomor_perkara",
        "label"
    ]]

In [17]:
query = """
Penggugat membeli tanah dari tergugat
namun sertifikat belum dibalik nama.
"""

retrieve(query)

,case_id,nomor_perkara,label
7,8,192/Pdt.G/2020/PN,Dikabulkan Sebagian
15,16,227/Pdt.G/2020/PN,Dikabulkan Sebagian
12,13,207/Pdt.G/2020/PN,Dikabulkan Sebagian
6,7,185/Pdt.G/2020/PN,Dikabulkan Sebagian
9,10,197/Pdt.G/2020/PN,Dikabulkan Sebagian


In [18]:
import joblib

joblib.dump(
    vectorizer,
    "../data/processed/tfidf_vectorizer.pkl"
)

['../data/processed/tfidf_vectorizer.pkl']